# Models and Loss Functions


In [ ]:
if "config" not in globals():
    raise RuntimeError("Run 00_experiment_runner.ipynb first so config is defined.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import torch
import torch.nn as nn
import os
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller
from torch.utils.data import DataLoader, TensorDataset, Dataset

## Model Definitions


In [ ]:
import math
import torch
import torch.nn as nn


def reshape_forecast(raw_output, batch_size, output_len, target_dim):
    return raw_output.view(batch_size, output_len, target_dim)


class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_len, target_dim):
        super().__init__()
        self.output_len = output_len
        self.target_dim = target_dim
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity="tanh",
        )
        self.fc = nn.Linear(hidden_size, output_len * target_dim)

    def forward(self, x):
        out, _ = self.rnn(x)
        raw = self.fc(out[:, -1, :])
        return reshape_forecast(raw, x.size(0), self.output_len, self.target_dim)


class LSTM(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        output_len,
        target_dim,
        dropout=0.0,
    ):
        super().__init__()
        recurrent_dropout = dropout if num_layers > 1 else 0.0
        self.output_len = output_len
        self.target_dim = target_dim
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=recurrent_dropout,
        )
        self.fc = nn.Linear(hidden_size, output_len * target_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        raw = self.fc(out[:, -1, :])
        return reshape_forecast(raw, x.size(0), self.output_len, self.target_dim)


class GRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_len, target_dim):
        super().__init__()
        self.output_len = output_len
        self.target_dim = target_dim
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, output_len * target_dim)

    def forward(self, x):
        out, _ = self.gru(x)
        raw = self.fc(out[:, -1, :])
        return reshape_forecast(raw, x.size(0), self.output_len, self.target_dim)


class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_len, target_dim):
        super().__init__()
        self.output_len = output_len
        self.target_dim = target_dim
        self.bilstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(hidden_size * 2, output_len * target_dim)

    def forward(self, x):
        out, _ = self.bilstm(x)
        raw = self.fc(out[:, -1, :])
        return reshape_forecast(raw, x.size(0), self.output_len, self.target_dim)


class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.0):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(
            in_channels, out_channels, kernel_size,
            padding=padding, dilation=dilation,
        )
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(
            out_channels, out_channels, kernel_size,
            padding=padding, dilation=dilation,
        )
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.downsample = (
            nn.Conv1d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels else None
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.dropout1(self.relu1(self.chomp1(self.conv1(x))))
        out = self.dropout2(self.relu2(self.chomp2(self.conv2(out))))
        residual = x if self.downsample is None else self.downsample(x)
        return self.relu(out + residual)


class TCN(nn.Module):
    def __init__(
        self,
        input_size,
        output_len,
        target_dim,
        channels=(32, 32, 64),
        kernel_size=3,
        dropout=0.1,
    ):
        super().__init__()
        self.output_len = output_len
        self.target_dim = target_dim
        layers = []
        in_channels = input_size
        for i, out_channels in enumerate(channels):
            layers.append(
                TemporalBlock(
                    in_channels,
                    out_channels,
                    kernel_size,
                    dilation=2 ** i,
                    dropout=dropout,
                )
            )
            in_channels = out_channels
        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(channels[-1], output_len * target_dim)

    def forward(self, x):
        out = self.network(x.transpose(1, 2))
        raw = self.fc(out[:, :, -1])
        return reshape_forecast(raw, x.size(0), self.output_len, self.target_dim)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model > 1:
            pe[:, 1::2] = torch.cos(position * div_term[:pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TransformerModel(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        output_len,
        target_dim,
        nhead=2,
        dim_feedforward=64,
        dropout=0.0,
    ):
        super().__init__()
        if hidden_size % nhead != 0:
            raise ValueError("hidden_size must be divisible by nhead")

        self.output_len = output_len
        self.target_dim = target_dim
        self.input_proj = nn.Linear(input_size, hidden_size)
        self.pos_encoder = PositionalEncoding(hidden_size)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="relu",
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )
        self.fc = nn.Linear(hidden_size, output_len * target_dim)

    def forward(self, x):
        z = self.pos_encoder(self.input_proj(x))
        out = self.transformer(z)
        raw = self.fc(out[:, -1, :])
        return reshape_forecast(raw, x.size(0), self.output_len, self.target_dim)


In [ ]:
def build_model(config):
    model_name = config.get("model_name", "LSTM")
    common = {
        "input_size": config["input_dim"],
        "output_len": config["model_output_len"],
        "target_dim": config["target_dim"],
    }

    if model_name == "RNN":
        return RNN(
            **common,
            hidden_size=config["hidden_size"],
            num_layers=config["num_layers"],
        )

    if model_name == "LSTM":
        return LSTM(
            **common,
            hidden_size=config["hidden_size"],
            num_layers=config["num_layers"],
            dropout=config.get("dropout", 0.0),
        )

    if model_name == "GRU":
        return GRU(
            **common,
            hidden_size=config["hidden_size"],
            num_layers=config["num_layers"],
        )

    if model_name == "BiLSTM":
        return BiLSTM(
            **common,
            hidden_size=config["hidden_size"],
            num_layers=config["num_layers"],
        )

    if model_name == "TCN":
        return TCN(
            **common,
            channels=config.get("tcn_channels", [32, 32, 64]),
            kernel_size=config.get("tcn_kernel_size", 3),
            dropout=config.get("tcn_dropout", 0.1),
        )

    if model_name == "Transformer":
        return TransformerModel(
            **common,
            hidden_size=config["hidden_size"],
            num_layers=config["num_layers"],
            nhead=config["nhead"],
            dim_feedforward=config["dim_feedforward"],
            dropout=config["dropout"],
        )

    raise ValueError(f"Unknown model_name: {model_name}")


## Loss Definitions


In [ ]:
import copy
import json
import os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader


def get_experiment_dir(config):
    experiment_dir = config.get("experiment_dir")
    if experiment_dir is None:
        experiment_name = config.get("experiment_name", "default_experiment")
        experiment_dir = os.path.join(
            config.get("results_dir", "./results_ar_eval_200"),
            experiment_name,
        )
    os.makedirs(experiment_dir, exist_ok=True)
    return experiment_dir


def get_checkpoint_dir(config):
    ckpt_dir = config.get("checkpoint_dir")
    if ckpt_dir is None:
        experiment_name = config.get("experiment_name", "default_experiment")
        ckpt_dir = os.path.join(
            config.get("checkpoint_root", "./loss_checkpoints"),
            experiment_name,
        )
    os.makedirs(ckpt_dir, exist_ok=True)
    return ckpt_dir


def make_json_safe(obj):
    safe = {}
    for key, value in obj.items():
        if key == "model":
            continue
        if isinstance(value, (str, int, float, bool)) or value is None:
            safe[key] = value
        elif isinstance(value, (list, tuple)):
            safe[key] = list(value)
        elif isinstance(value, dict):
            safe[key] = make_json_safe(value)
        else:
            safe[key] = str(value)
    return safe


def save_checkpoint(model, config, model_name, tag, epoch, val_loss):
    ckpt_dir = get_checkpoint_dir(config)
    path = os.path.join(ckpt_dir, f"{model_name}_{tag}.pt")

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "model_name": model_name,
            "tag": tag,
            "epoch": int(epoch),
            "val_loss": float(val_loss),
            "config": make_json_safe(config),
            "base_input_cols": list(base_input_cols),
            "pre_pca_input_cols": list(pre_pca_input_cols),
            "model_input_cols": list(model_input_cols),
            "target_cols": list(target_cols),
            "report_col": report_col,
            "loss_type": config.get("loss_type", "mse"),
        },
        path,
    )
    return path


In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    import soft_dtw
    import path_soft_dtw
except Exception as e:
    soft_dtw = None
    path_soft_dtw = None
    print("DILATE loss 모듈을 불러오지 못했습니다. loss_type=\"dilate\"일 때만 필요합니다.")
    print("원인:", repr(e))


# ============================================================
# Utility
# ============================================================

def ensure_3d(pred, target):
    """
    pred/target을 모두 (B, T, C) 형태로 통일.
    """
    if pred.dim() == 2:
        pred = pred.unsqueeze(-1)
    if target.dim() == 2:
        target = target.unsqueeze(-1)
    return pred, target


# ============================================================
# 1. MSE Loss
# ============================================================

def mse_loss(pred, target):
    pred, target = ensure_3d(pred, target)
    return F.mse_loss(pred, target)


# ============================================================
# 2. Original EP Loss
# ============================================================

class EPLoss(nn.Module):
    def __init__(self, threshold=0.8, factor_under=2.0, factor_over=1.6):
        super(EPLoss, self).__init__()
        self.T = threshold
        self.F_u = factor_under
        self.F_o = factor_over

    def forward(self, y_pred, y_true):
        # 1. 기본 MSE 오차 계산
        base_loss = (y_pred - y_true) ** 2

        # 2. 조건 마스크 생성
        peak_mask = (y_true > self.T).float()

        # 3. 과소평가 / 과대평가 마스크 생성
        under_mask = (y_true > y_pred).float()
        over_mask = (y_pred > y_true).float()

        # 4. 개별 페널티 항 계산
        under_penalty = base_loss * self.F_u * peak_mask * under_mask
        over_penalty = base_loss * self.F_o * peak_mask * over_mask

        # 5. 최종 Loss
        total_loss = base_loss + under_penalty + over_penalty

        return torch.mean(total_loss)


def ep_loss(pred, target, threshold=0.8, factor_under=2.0, factor_over=1.6):
    pred, target = ensure_3d(pred, target)

    criterion = EPLoss(
        threshold=threshold,
        factor_under=factor_under,
        factor_over=factor_over,
    )

    return criterion(pred, target)


# ============================================================
# 3. Original DILATE Loss
# ============================================================

def dilate_loss(outputs, targets, alpha, gamma, device):
    """
    outputs, targets: shape (batch_size, N_output, 1)
    """
    outputs, targets = ensure_3d(outputs, targets)

    if soft_dtw is None or path_soft_dtw is None:
        raise ImportError("DILATE loss를 사용하려면 soft_dtw.py와 path_soft_dtw.py가 필요합니다.")

    batch_size, N_output, num_channels = outputs.shape

    softdtw_batch = soft_dtw.SoftDTWBatch.apply

    D = torch.zeros((batch_size, N_output, N_output)).to(device)

    for k in range(batch_size):
        Dk = 0.0
        for channel in range(num_channels):
            Dk = Dk + soft_dtw.pairwise_distances(
                targets[k, :, channel].view(-1, 1),
                outputs[k, :, channel].view(-1, 1),
            )
        D[k:k + 1, :, :] = Dk / num_channels

    loss_shape = softdtw_batch(D, gamma)

    path_dtw = path_soft_dtw.PathDTWBatch.apply
    path = path_dtw(D, gamma)

    Omega = soft_dtw.pairwise_distances(
        torch.arange(1, N_output + 1).view(N_output, 1)
    ).to(device)

    loss_temporal = torch.sum(path * Omega) / (N_output * N_output)

    loss = alpha * loss_shape + (1 - alpha) * loss_temporal

    return loss, loss_shape, loss_temporal


# ============================================================
# 4. Original TILDE-Q Loss
# ============================================================

PI = 3.141592653589793


def amp_loss(outputs, targets):
    # outputs = B, T, 1 --> B, 1, T
    B, _, T = outputs.shape

    fft_size = 1 << (2 * T - 1).bit_length()

    out_fourier = torch.fft.fft(outputs, fft_size, dim=-1)
    tgt_fourier = torch.fft.fft(targets, fft_size, dim=-1)

    out_norm = torch.norm(outputs, dim=-1, keepdim=True)
    tgt_norm = torch.norm(targets, dim=-1, keepdim=True)

    # calculate normalized auto correlation
    auto_corr = torch.fft.ifft(
        tgt_fourier * tgt_fourier.conj(),
        dim=-1,
    ).real

    auto_corr = torch.cat(
        [auto_corr[..., -(T - 1):], auto_corr[..., :T]],
        dim=-1,
    )

    nac_tgt = auto_corr / (tgt_norm * tgt_norm)

    # calculate cross correlation
    cross_corr = torch.fft.ifft(
        tgt_fourier * out_fourier.conj(),
        dim=-1,
    ).real

    cross_corr = torch.cat(
        [cross_corr[..., -(T - 1):], cross_corr[..., :T]],
        dim=-1,
    )

    nac_out = cross_corr / (tgt_norm * out_norm)

    loss = torch.mean(torch.abs(nac_tgt - nac_out))

    return loss


def ashift_loss(outputs, targets):
    B, _, T = outputs.shape

    return T * torch.mean(
        torch.abs(1 / T - torch.softmax(outputs - targets, dim=-1))
    )


def phase_loss(outputs, targets):
    B, _, T = outputs.shape

    out_fourier = torch.fft.fft(outputs, dim=-1)
    tgt_fourier = torch.fft.fft(targets, dim=-1)

    tgt_fourier_sq = tgt_fourier.real ** 2 + tgt_fourier.imag ** 2

    mask = (tgt_fourier_sq > T).float()

    topk_indices = tgt_fourier_sq.topk(
        k=int(T ** 0.5),
        dim=-1,
    ).indices

    mask = mask.scatter_(-1, topk_indices, 1.0)
    mask[..., 0] = 1.0

    mask = torch.where(mask > 0, 1.0, 0.0)
    mask = mask.bool()

    not_mask = (~mask).float()
    not_mask /= torch.mean(not_mask)

    out_fourier_sq = torch.abs(out_fourier.real) + torch.abs(out_fourier.imag)

    zero_error = torch.abs(out_fourier) * not_mask
    zero_error = torch.where(
        torch.isnan(zero_error),
        torch.zeros_like(zero_error),
        zero_error,
    )

    mask = mask.float()
    mask /= torch.mean(mask)

    ae = torch.abs(out_fourier - tgt_fourier) * mask
    ae = torch.where(
        torch.isnan(ae),
        torch.zeros_like(ae),
        ae,
    )

    phase_loss_value = (torch.mean(zero_error) + torch.mean(ae)) / (T ** 0.5)

    return phase_loss_value


def tildeq_loss(outputs, targets, alpha=0.5, gamma=0.0, beta=0.5):
    outputs, targets = ensure_3d(outputs, targets)

    outputs = outputs.permute(0, 2, 1)
    targets = targets.permute(0, 2, 1)

    assert not torch.isnan(outputs).any(), "Nan value detected!"
    assert not torch.isinf(outputs).any(), "Inf value detected!"

    B, _, T = outputs.shape

    l_ashift = ashift_loss(outputs, targets)
    l_amp = amp_loss(outputs, targets)
    l_phase = phase_loss(outputs, targets)

    loss = alpha * l_ashift + (1 - alpha) * l_phase + gamma * l_amp

    assert loss == loss, "Loss Nan!"

    return loss


# ============================================================
# 5. Original DBLoss
# ============================================================

class EMA(nn.Module):
    """
    Exponential Moving Average (EMA) block to highlight the trend of time series
    """

    def __init__(self, alpha):
        super(EMA, self).__init__()
        self.alpha = alpha

    def forward(self, x):
        _, t, _ = x.shape

        powers = torch.flip(
            torch.arange(t, dtype=x.dtype, device=x.device),
            dims=(0,),
        )
        weights = torch.pow(1 - self.alpha, powers)
        divisor = weights.clone()

        weights[1:] = weights[1:] * self.alpha

        weights = weights.reshape(1, t, 1)
        divisor = divisor.reshape(1, t, 1)

        x = torch.cumsum(x * weights, dim=1)
        x = torch.div(x, divisor)

        return x.to(dtype=x.dtype)


class DECOMP(nn.Module):
    """
    Series decomposition block
    """

    def __init__(self, alpha):
        super(DECOMP, self).__init__()
        self.ma = EMA(alpha)

    def forward(self, x):
        moving_average = self.ma(x)
        res = x - moving_average
        return res, moving_average


class DBLoss(nn.Module):
    """
    Decomposition-based loss.
    trend + season dual loss.
    """

    def __init__(self, alpha, beta):
        super().__init__()
        self.decomp = DECOMP(alpha)
        self.beta = beta
        self.mse = nn.MSELoss(reduction="mean")
        self.mae = nn.L1Loss(reduction="mean")

    def forward(self, pred, target):
        pred_season, pred_trend = self.decomp(pred)
        target_season, target_trend = self.decomp(target)

        season_loss = self.mse(pred_season, target_season)
        trend_loss = self.mae(pred_trend, target_trend)

        trend_loss = trend_loss * (
            season_loss / (trend_loss + 1e-8)
        ).detach()

        return self.beta * season_loss + (1 - self.beta) * trend_loss


def db_loss(pred, target, alpha=0.3, beta=0.5):
    pred, target = ensure_3d(pred, target)

    criterion = DBLoss(
        alpha=alpha,
        beta=beta,
    )

    return criterion(pred, target)


# ============================================================
# 6. Original KMBDF Loss
# ============================================================

class KMBDFLoss(nn.Module):
    def __init__(
        self,
        auxi_mode="rfft",
        auxi_type="complex",
        auxi_loss_type="mse",
        auxi_lambda=1.0,
        joint_forecast=True,
    ):
        """
        KMB-DF Loss Module

        Args:
            auxi_mode: 'fft' 또는 'rfft'
            auxi_type: 'complex', 'real', 'imag', 'abs', 'angle'
            auxi_loss_type: 'mse' 또는 'mae'
            auxi_lambda: KMB Loss 가중치
            joint_forecast: 과거 시점 X와 예측 시점 Y를 붙여서 계산할지 여부
        """
        super(KMBDFLoss, self).__init__()

        self.auxi_mode = auxi_mode
        self.auxi_type = auxi_type
        self.auxi_loss_type = auxi_loss_type
        self.auxi_lambda = auxi_lambda
        self.joint_forecast = joint_forecast

        if self.auxi_loss_type == "mse":
            self.criterion = nn.MSELoss()
        elif self.auxi_loss_type == "mae":
            self.criterion = nn.L1Loss()
        else:
            self.criterion = nn.MSELoss()

    def forward(self, outputs, batch_y, batch_x=None):
        """
        Args:
            outputs: 모델 예측값 [Batch, Pred_Len, Features]
            batch_y: 실제 정답값 [Batch, Pred_Len, Features]
            batch_x: 입력 과거 데이터 [Batch, Seq_Len, Features]
        """

        # 1. Joint Distribution 정렬 옵션 처리
        if self.joint_forecast and batch_x is not None:
            outputs = torch.concat(
                (batch_x.to(outputs.device), outputs),
                dim=1,
            ).float()

            batch_y = torch.concat(
                (batch_x.to(batch_y.device), batch_y),
                dim=1,
            ).float()

        # 2. FFT / RFFT 주파수 공간상에서의 오차 변환
        if self.auxi_mode == "fft":
            loss_auxi = torch.fft.fft(outputs, dim=1) - torch.fft.fft(batch_y, dim=1)

        elif self.auxi_mode == "rfft":
            if self.auxi_type == "complex":
                loss_auxi = torch.fft.rfft(outputs, dim=1) - torch.fft.rfft(batch_y, dim=1)

            elif self.auxi_type == "real":
                loss_auxi = torch.fft.rfft(outputs, dim=1).real - torch.fft.rfft(batch_y, dim=1).real

            elif self.auxi_type == "imag":
                loss_auxi = torch.fft.rfft(outputs, dim=1).imag - torch.fft.rfft(batch_y, dim=1).imag

            elif self.auxi_type == "abs":
                loss_auxi = torch.abs(torch.fft.rfft(outputs, dim=1)) - torch.abs(torch.fft.rfft(batch_y, dim=1))

            elif self.auxi_type == "angle":
                loss_auxi = torch.angle(torch.fft.rfft(outputs, dim=1)) - torch.angle(torch.fft.rfft(batch_y, dim=1))

        else:
            return torch.tensor(0.0, device=outputs.device)

        # 3. 오차 최종 계산
        if self.auxi_loss_type == "mse":
            if loss_auxi.is_complex():
                # For complex tensors, apply MSE to real and imaginary parts separately
                loss_real = self.criterion(loss_auxi.real, torch.zeros_like(loss_auxi.real).to(loss_auxi.device))
                loss_imag = self.criterion(loss_auxi.imag, torch.zeros_like(loss_auxi.imag).to(loss_auxi.device))
                loss_auxi = loss_real + loss_imag # Sum of MSE for real and imaginary parts
            else:
                loss_auxi = self.criterion(
                    loss_auxi,
                    torch.zeros_like(loss_auxi).to(loss_auxi.device),
                )

        elif self.auxi_loss_type == "mae":
            loss_auxi = torch.abs(loss_auxi).mean()

        return self.auxi_lambda * loss_auxi


def kmbdf_loss(
    pred,
    target,
    batch_x=None,
    auxi_mode="rfft",
    auxi_type="complex",
    auxi_loss_type="mse",
    auxi_lambda=1.0,
    joint_forecast=True,
):
    pred, target = ensure_3d(pred, target)

    if batch_x is not None and batch_x.dim() == 2:
        batch_x = batch_x.unsqueeze(-1)

    criterion = KMBDFLoss(
        auxi_mode=auxi_mode,
        auxi_type=auxi_type,
        auxi_loss_type=auxi_loss_type,
        auxi_lambda=auxi_lambda,
        joint_forecast=joint_forecast,
    )

    return criterion(pred, target, batch_x=batch_x)


# ============================================================
# 7. Original CP Loss
# ============================================================

class PerceptualFilter(nn.Module):
    def __init__(self, channels, levels=4):
        super().__init__()
        self.levels = levels

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=channels,
                out_channels=channels,
                kernel_size=5,
                padding=2,
                groups=channels,
            )
            for _ in range(levels)
        ])

    def forward(self, x):
        # [Batch, Length, Channel] -> [Batch, Channel, Length]
        curr_x = x.permute(0, 2, 1)

        features = []

        for i in range(self.levels):
            down = F.interpolate(
                curr_x,
                scale_factor=0.5,
                mode="linear",
                align_corners=False,
            )

            smoothed = self.convs[i](down)

            up = F.interpolate(
                smoothed,
                size=curr_x.shape[-1],
                mode="linear",
                align_corners=False,
            )

            detail = curr_x - up
            features.append(detail)

            curr_x = up

        features.append(curr_x)

        return features


class CPLoss(nn.Module):
    def __init__(self, num_channels, levels=4):
        super().__init__()
        self.p_filter = PerceptualFilter(num_channels, levels)

    def forward(self, y_pred, y_true):
        feats_pred = self.p_filter(y_pred)
        feats_true = self.p_filter(y_true)

        loss = 0

        for f_p, f_t in zip(feats_pred, feats_true):
            loss += F.l1_loss(f_p, f_t)

        return loss


def cp_loss(pred, target, num_channels=None, levels=4):
    pred, target = ensure_3d(pred, target)

    if num_channels is None:
        num_channels = pred.shape[-1]

    criterion = CPLoss(
        num_channels=num_channels,
        levels=levels,
    ).to(pred.device)

    return criterion(pred, target)


# ============================================================
# compute_loss
# ============================================================

def compute_loss(pred, target, config, batch_x=None):
    """
    단독 원본 loss만 지원.

    지원 loss_type:
    - mse
    - ep
    - dilate
    - tildeq
    - db
    - kmbdf
    - cp
    """
    loss_type = config.get("loss_type", "mse")

    if loss_type == "mse":
        return mse_loss(pred, target)

    elif loss_type == "ep":
        return ep_loss(
            pred,
            target,
            threshold=config.get("threshold", 0.8),
            factor_under=config.get("factor_under", 2.0),
            factor_over=config.get("factor_over", 1.6),
        )

    elif loss_type == "dilate":
        loss, loss_shape, loss_temporal = dilate_loss(
            pred,
            target,
            alpha=config.get("alpha", 0.5),
            gamma=config.get("gamma", 0.01),
            device=config.get("device", pred.device),
        )
        return loss

    elif loss_type == "tildeq":
        return tildeq_loss(
            pred,
            target,
            alpha=config.get("alpha", 0.5),
            gamma=config.get("gamma_tildeq", 0.0),
            beta=config.get("beta", 0.5),
        )

    elif loss_type == "db":
        return db_loss(
            pred,
            target,
            alpha=config.get("db_alpha", 0.3),
            beta=config.get("db_beta", 0.5),
        )

    elif loss_type == "kmbdf":
        return kmbdf_loss(
            pred,
            target,
            batch_x=batch_x,
            auxi_mode=config.get("auxi_mode", "rfft"),
            auxi_type=config.get("auxi_type", "complex"),
            auxi_loss_type=config.get("auxi_loss_type", "mse"),
            auxi_lambda=config.get("auxi_lambda", 1.0),
            joint_forecast=config.get("joint_forecast", True),
        )

    elif loss_type == "cp":
        return cp_loss(
            pred,
            target,
            num_channels=config.get("num_channels", None),
            levels=config.get("cp_levels", 4),
        )

    else:
        raise ValueError(f"Unknown loss_type: {loss_type}")


## Model and Loss Creation
